## Import Libraries

In [17]:
from pathlib import Path
from collections import Counter
import os
import random
import shutil
import time
import json

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

try:
    from ultralytics import YOLO
    ULTRALYTICS_AVAILABLE = True
except Exception as error:
    ULTRALYTICS_AVAILABLE = False
    print("Ultralytics import failed:", error)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

EXPECTED_CLASSES = 82

print("LIBRARIES AND GLOBAL SETTINGS")
print("Random seed:", SEED)
print("Expected classes:", EXPECTED_CLASSES)
print("Ultralytics available:", ULTRALYTICS_AVAILABLE)


LIBRARIES AND GLOBAL SETTINGS
Random seed: 42
Expected classes: 82
Ultralytics available: True


E:\CondaEnvs\smartagrivision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset Paths

In [18]:
PROJECT_ROOT = Path(
    r"E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks"
)

CLEANED_DATASET_ROOT = (
    PROJECT_ROOT / "Cleaned_Dataset"
)

PREPROCESSED_DATASET_ROOT = (
    PROJECT_ROOT / "Preprocessed_Dataset"
)

METADATA_ROOT = (
    PREPROCESSED_DATASET_ROOT / "metadata"
)

PREPROCESSED_METADATA_PATH = (
    METADATA_ROOT / "image_metadata_preprocessed.csv"
)

YOLO_CLASSIFICATION_ROOT = (
    PROJECT_ROOT / "YOLO_Classification_Dataset"
)

MODEL_OUTPUT_ROOT = (
    PROJECT_ROOT / "YOLO_Models"
)

TRAINING_RUNS_ROOT = (
    MODEL_OUTPUT_ROOT / "classification_runs"
)

print("DATASET PATHS")
print("Project Root:", PROJECT_ROOT)
print("Cleaned Dataset:", CLEANED_DATASET_ROOT)
print("Preprocessed Dataset:", PREPROCESSED_DATASET_ROOT)
print("Metadata:", PREPROCESSED_METADATA_PATH)
print("YOLO Dataset:", YOLO_CLASSIFICATION_ROOT)
print("Model Output:", MODEL_OUTPUT_ROOT)


DATASET PATHS
Project Root: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks
Cleaned Dataset: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Cleaned_Dataset
Preprocessed Dataset: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Preprocessed_Dataset
Metadata: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\Preprocessed_Dataset\metadata\image_metadata_preprocessed.csv
YOLO Dataset: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Classification_Dataset
Model Output: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Models


In [19]:
print("PATH EXISTENCE CHECK")
print("Project exists:", PROJECT_ROOT.exists())
print("Cleaned Dataset exists:", CLEANED_DATASET_ROOT.exists())
print("Preprocessed Dataset exists:", PREPROCESSED_DATASET_ROOT.exists())
print("Metadata exists:", PREPROCESSED_METADATA_PATH.exists())
print("YOLO Dataset exists:", YOLO_CLASSIFICATION_ROOT.exists())
print("Model Output exists:", MODEL_OUTPUT_ROOT.exists())


PATH EXISTENCE CHECK
Project exists: True
Cleaned Dataset exists: True
Preprocessed Dataset exists: True
Metadata exists: True
YOLO Dataset exists: False
Model Output exists: True


##  Load Preprocessed Image Metadata

In [20]:
if not PREPROCESSED_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Required metadata file was not found:\n"
        f"{PREPROCESSED_METADATA_PATH}\n\n"
        "Run 05_Image_Preprocessing.ipynb first."
    )

image_metadata_df = pd.read_csv(
    PREPROCESSED_METADATA_PATH
)

print("IMAGE METADATA LOADED")
print("Rows:", f"{len(image_metadata_df):,}")
print("Columns:", len(image_metadata_df.columns))
display(image_metadata_df.head())


IMAGE METADATA LOADED
Rows: 294,695
Columns: 8


,Dataset,Split,Class,Image_Path,Filename,Label,Encoded_Label,Processed_Split
0,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_103_100.jpg,79,79,Unsplit
1,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_107_100.jpg,79,79,Unsplit
2,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_111_100.jpg,79,79,Unsplit
3,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_115_100.jpg,79,79,Unsplit
4,Fruits-360,Unsplit,fruits-360_100x100,Cleaned_Dataset\Fruits-360\fruits-360_100x100\...,r0_119_100.jpg,79,79,Unsplit


In [21]:
required_columns = {
    "Dataset",
    "Split",
    "Class",
    "Image_Path",
    "Filename",
    "Label"
}

missing_columns = (
    required_columns
    - set(image_metadata_df.columns)
)

print("METADATA STRUCTURE CHECK")

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

print("Required columns: OK")
print("Total images:", f"{len(image_metadata_df):,}")
print("Total datasets:", image_metadata_df["Dataset"].nunique())
print("Total classes:", image_metadata_df["Class"].nunique())
print()
print("Missing values:")
display(image_metadata_df[list(required_columns)].isnull().sum())


METADATA STRUCTURE CHECK
Required columns: OK
Total images: 294,695
Total datasets: 4
Total classes: 82

Missing values:


Dataset       0
Label         0
Filename      0
Class         0
Image_Path    0
Split         0
dtype: int64

##  Image Path Resolution

In [22]:
# The previous notebooks store Image_Path values that may be
# absolute or relative. This resolver checks several project roots.

PATH_ROOT_CANDIDATES = [
    PROJECT_ROOT,
    PROJECT_ROOT.parent,
    CLEANED_DATASET_ROOT,
    PREPROCESSED_DATASET_ROOT
]

def resolve_image_path(value):
    if pd.isna(value):
        return None

    raw = Path(str(value).strip())

    if raw.is_absolute() and raw.exists():
        return raw

    candidates = []

    for root in PATH_ROOT_CANDIDATES:
        candidates.append(root / raw)

    # Also support Windows-style relative paths when the current
    # metadata was generated from the notebooks directory.
    candidates.append(PROJECT_ROOT / raw)

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    return None

resolved_paths = []

for value in tqdm(
    image_metadata_df["Image_Path"],
    desc="Resolving image paths",
    unit="image"
):
    resolved_paths.append(
        resolve_image_path(value)
    )

image_metadata_df["Resolved_Image_Path"] = resolved_paths

resolved_count = (
    image_metadata_df["Resolved_Image_Path"]
    .notna()
    .sum()
)

print("IMAGE PATH RESOLUTION")
print("Metadata rows:", f"{len(image_metadata_df):,}")
print("Resolved images:", f"{resolved_count:,}")
print(
    "Unresolved images:",
    f"{len(image_metadata_df) - resolved_count:,}"
)


Resolving image paths: 100%|██████████| 294695/294695 [03:19<00:00, 1480.43image/s]


IMAGE PATH RESOLUTION
Metadata rows: 294,695
Resolved images: 294,695
Unresolved images: 0


In [23]:
if resolved_count == 0:
    raise RuntimeError(
        "No image paths could be resolved. "
        "Check the Image_Path values in image_metadata_preprocessed.csv."
    )

unresolved_sample = (
    image_metadata_df[
        image_metadata_df["Resolved_Image_Path"].isna()
    ][
        ["Dataset", "Class", "Image_Path", "Filename"]
    ]
    .head(20)
)

if len(unresolved_sample):
    print("SAMPLE UNRESOLVED PATHS")
    display(unresolved_sample)
else:
    print("All metadata image paths were resolved.")


All metadata image paths were resolved.


##  Class / Label Configuration

In [24]:
class_names = sorted(
    image_metadata_df["Class"]
    .astype(str)
    .str.strip()
    .unique()
)

NUM_CLASSES = len(class_names)

class_to_id = {
    class_name: index
    for index, class_name in enumerate(class_names)
}

image_metadata_df["YOLO_Class_ID"] = (
    image_metadata_df["Class"]
    .astype(str)
    .str.strip()
    .map(class_to_id)
)

print("CLASS CONFIGURATION")
print("Number of classes:", NUM_CLASSES)

if NUM_CLASSES != EXPECTED_CLASSES:
    raise ValueError(
        f"Expected {EXPECTED_CLASSES} classes, "
        f"but metadata contains {NUM_CLASSES}."
    )

print("82-class validation: PASSED")
print()
print("First 10 class mappings:")
display(
    pd.DataFrame({
        "Class": class_names[:10],
        "YOLO_ID": list(range(10))
    })
)


CLASS CONFIGURATION
Number of classes: 82
82-class validation: PASSED

First 10 class mappings:


,Class,YOLO_ID
0,Apple_Scab_Leaf,0
1,Apple___Apple_scab,1
2,Apple___Black_rot,2
3,Apple___Cedar_apple_rust,3
4,Apple___healthy,4
5,Apple_leaf,5
6,Apple_rust_leaf,6
7,Bean,7
8,Bell_pepper_leaf,8
9,Bell_pepper_leaf_spot,9


##  Source Image Validation

In [25]:
def is_valid_image(path):
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except Exception:
        return False

# Validate only unresolved/resolved status first.
# Full image verification is intentionally performed during dataset creation.
source_ready_df = (
    image_metadata_df[
        image_metadata_df["Resolved_Image_Path"].notna()
    ]
    .copy()
)

print("SOURCE IMAGE VALIDATION")
print("Images available for YOLO preparation:", f"{len(source_ready_df):,}")


SOURCE IMAGE VALIDATION
Images available for YOLO preparation: 294,695


In [26]:
class_counts_source = (
    source_ready_df
    .groupby("Class")
    .size()
    .sort_values()
)

print("SOURCE CLASS DISTRIBUTION")
print("Minimum images in one class:", int(class_counts_source.min()))
print("Maximum images in one class:", int(class_counts_source.max()))
print("Classes:", len(class_counts_source))

if len(class_counts_source) != EXPECTED_CLASSES:
    raise ValueError(
        "Resolved images do not contain all 82 classes."
    )


SOURCE CLASS DISTRIBUTION
Minimum images in one class: 2
Maximum images in one class: 182942
Classes: 82


##  Train / Validation / Test Split Preparation

In [27]:
# We build the YOLO classification split from the complete resolved
# metadata instead of trusting an incomplete previous YOLO folder.
#
# Every class receives:
#   80% train
#   10% validation
#   10% test
#
# This avoids the earlier problem where validation had only 54 classes
# and test had only 43 classes.

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

print("SPLIT CONFIGURATION")
print("Train:", TRAIN_RATIO)
print("Validation:", VAL_RATIO)
print("Test:", TEST_RATIO)


SPLIT CONFIGURATION
Train: 0.8
Validation: 0.1
Test: 0.1


In [28]:
rng = random.Random(SEED)

split_records = []

for class_name in class_names:

    class_df = (
        source_ready_df[
            source_ready_df["Class"].astype(str).str.strip()
            == class_name
        ]
        .copy()
        .reset_index(drop=True)
    )

    indices = list(range(len(class_df)))
    rng.shuffle(indices)

    n = len(indices)

    n_train = max(1, int(n * TRAIN_RATIO))
    n_val = max(1, int(n * VAL_RATIO))

    # Keep at least one image for test.
    if n_train + n_val >= n:
        n_val = max(1, min(n_val, n - n_train - 1))

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train + n_val]
    test_idx = indices[n_train + n_val:]

    assignments = {
        "Train": train_idx,
        "Validation": val_idx,
        "Test": test_idx
    }

    for split_name, selected_indices in assignments.items():

        for idx in selected_indices:

            row = class_df.iloc[idx]

            split_records.append({
                "Dataset": row["Dataset"],
                "Class": class_name,
                "Label": int(class_to_id[class_name]),
                "Image_Path": row["Image_Path"],
                "Resolved_Image_Path": row["Resolved_Image_Path"],
                "Filename": row["Filename"],
                "Split": split_name
            })

yolo_split_df = pd.DataFrame(
    split_records
)

print("SPLIT PREPARATION COMPLETED")
print("Rows:", f"{len(yolo_split_df):,}")
display(
    yolo_split_df["Split"].value_counts()
)


SPLIT PREPARATION COMPLETED
Rows: 294,695


Split
Train         235731
Test           29521
Validation     29443
Name: count, dtype: int64

## Split Validation

In [31]:
split_class_table = (
    yolo_split_df
    .groupby(["Split", "Class"])
    .size()
    .unstack(fill_value=0)
)

print("SPLIT / CLASS VALIDATION")

for split_name in ["Train", "Validation", "Test"]:

    if split_name not in split_class_table.index:

        print(
            split_name,
            "split: NOT AVAILABLE"
        )

        continue

    missing = [
        class_name
        for class_name in class_names
        if split_class_table.loc[split_name, class_name] == 0
    ]

    present = (
        len(class_names)
        - len(missing)
    )

    print(
        split_name,
        "classes:",
        present,
        "/",
        NUM_CLASSES
    )

    if missing:

        print(
            "Missing classes:",
            missing[:10]
        )

print("SPLIT / CLASS VALIDATION COMPLETED")

SPLIT / CLASS VALIDATION
Train classes: 82 / 82
Validation classes: 82 / 82
Test classes: 81 / 82
Missing classes: ['Tomato_two_spotted_spider_mites_leaf']
SPLIT / CLASS VALIDATION COMPLETED


In [32]:
split_summary = (
    yolo_split_df
    .groupby("Split")
    .size()
    .reindex(["Train", "Validation", "Test"])
    .reset_index(name="Images")
)

display(split_summary)


,Split,Images
0,Train,235731
1,Validation,29443
2,Test,29521


## YOLO Classification Dataset Structure

In [33]:
SPLIT_FOLDER_MAP = {
    "Train": "train",
    "Validation": "val",
    "Test": "test"
}

print("YOLO CLASSIFICATION STRUCTURE")

for split_folder in ["train", "val", "test"]:
    print(
        YOLO_CLASSIFICATION_ROOT
        / split_folder
        / "<class_name>"
    )


YOLO CLASSIFICATION STRUCTURE
E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Classification_Dataset\train\<class_name>
E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Classification_Dataset\val\<class_name>
E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Classification_Dataset\test\<class_name>


In [34]:
# Create class folders only after split validation passes.
# The original datasets remain untouched.

for split_folder in ["train", "val", "test"]:

    for class_name in class_names:

        (
            YOLO_CLASSIFICATION_ROOT
            / split_folder
            / class_name
        ).mkdir(
            parents=True,
            exist_ok=True
        )

print("YOLO classification folder structure created.")


YOLO classification folder structure created.


##  Image Linking / Copying

In [35]:
def place_image(source, target):
    # Hard links save a large amount of disk space when source and
    # destination are on the same NTFS volume.
    try:
        if target.exists():
            return "existing"

        os.link(
            source,
            target
        )

        return "hardlink"

    except Exception:
        try:
            if target.exists():
                return "existing"

            shutil.copy2(
                source,
                target
            )

            return "copy"

        except Exception as error:
            return f"failed: {error}"

print(
    "Image placement method ready: hardlink with copy fallback."
)


Image placement method ready: hardlink with copy fallback.


In [36]:
# Dataset creation can take time because many filesystem entries
# must be created. Existing files are skipped.

placement_counter = Counter()
failed_records = []

for row in tqdm(
    yolo_split_df.itertuples(index=False),
    total=len(yolo_split_df),
    desc="Preparing YOLO dataset",
    unit="image"
):

    source = Path(row.Resolved_Image_Path)

    split_folder = (
        SPLIT_FOLDER_MAP[row.Split]
    )

    target_dir = (
        YOLO_CLASSIFICATION_ROOT
        / split_folder
        / row.Class
    )

    target = (
        target_dir
        / row.Filename
    )

    result = place_image(
        source,
        target
    )

    placement_counter[
        result
    ] += 1

    if result.startswith("failed"):
        failed_records.append({
            "Source": str(source),
            "Target": str(target),
            "Error": result
        })

print("DATASET PREPARATION RESULT")
print(dict(placement_counter))
print("Failed:", len(failed_records))


Preparing YOLO dataset: 100%|██████████| 294695/294695 [01:57<00:00, 2510.15image/s]

DATASET PREPARATION RESULT
{'hardlink': 118110, 'existing': 176585}
Failed: 0


## YOLO Dataset Integrity Check

In [37]:
def count_split_class_images(root, split_folder, class_name):
    folder = (
        root
        / split_folder
        / class_name
    )

    if not folder.exists():
        return 0

    return sum(
        1
        for path in folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

final_counts = []

for split_name, split_folder in SPLIT_FOLDER_MAP.items():

    for class_name in class_names:

        count = count_split_class_images(
            YOLO_CLASSIFICATION_ROOT,
            split_folder,
            class_name
        )

        final_counts.append({
            "Split": split_name,
            "Class": class_name,
            "Images": count
        })

final_count_df = pd.DataFrame(
    final_counts
)

print("YOLO DATASET INTEGRITY")
display(
    final_count_df.groupby("Split")["Images"]
    .sum()
    .reindex(["Train", "Validation", "Test"])
    .to_frame()
)


YOLO DATASET INTEGRITY


,Images
Split,
Train,91926
Validation,13044
Test,13140


In [39]:
zero_class_rows = final_count_df[
    final_count_df["Images"] == 0
].copy()

if len(zero_class_rows) > 0:

    print("ZERO-IMAGE CLASS WARNING")

    display(
        zero_class_rows
    )

    print(
        "Warning: Some classes do not have "
        "images in every split."
    )

    print(
        "Training will continue because "
        "the class exists in the dataset."
    )

else:

    print(
        "All classes contain images "
        "in train/val/test."
    )

print(
    "CLASS IMAGE VALIDATION COMPLETED"
)

ZERO-IMAGE CLASS WARNING


,Split,Class,Images
242,Test,Tomato_two_spotted_spider_mites_leaf,0


Training will continue because the class exists in the dataset.
CLASS IMAGE VALIDATION COMPLETED


##  CPU-Friendly YOLO Configuration

In [40]:
YOLO_MODEL_NAME = "yolo11n-cls.pt"

# CPU-friendly first run
IMAGE_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 3

DEVICE = "cpu"
WORKERS = 0

# These are deliberately conservative for CPU.
# After the 3-epoch test succeeds, increase EPOCHS gradually.

print("YOLO CPU CONFIGURATION")
print("Model:", YOLO_MODEL_NAME)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Device:", DEVICE)
print("Workers:", WORKERS)


YOLO CPU CONFIGURATION
Model: yolo11n-cls.pt
Image size: 128
Batch size: 32
Epochs: 3
Device: cpu
Workers: 0


##  Load Pretrained YOLO Classification Model

In [41]:
model = None

if not ULTRALYTICS_AVAILABLE:

    print("MODEL LOAD BLOCKED")
    print("Ultralytics is not available.")

else:

    try:

        model = YOLO(
            YOLO_MODEL_NAME
        )

        print(
            "YOLO CLASSIFICATION MODEL LOADED"
        )

        print(
            "Model:",
            YOLO_MODEL_NAME
        )

    except Exception as error:

        print(
            "MODEL LOAD FAILED"
        )

        print(
            type(error).__name__,
            ":",
            error
        )


YOLO CLASSIFICATION MODEL LOADED
Model: yolo11n-cls.pt


##  Training Output Configuration

In [42]:
TRAINING_RUNS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

TRAIN_PROJECT = (
    TRAINING_RUNS_ROOT
)

TRAIN_NAME = (
    "smartagrivision_yolo_classification_cpu"
)

RUN_DIR = (
    TRAIN_PROJECT
    / TRAIN_NAME
)

print("TRAINING OUTPUT CONFIGURATION")
print("Project:", TRAIN_PROJECT)
print("Run name:", TRAIN_NAME)
print("Run directory:", RUN_DIR)


TRAINING OUTPUT CONFIGURATION
Project: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Models\classification_runs
Run name: smartagrivision_yolo_classification_cpu
Run directory: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Models\classification_runs\smartagrivision_yolo_classification_cpu


##  Final Training Safety Check

In [44]:
dataset_ready = (
    NUM_CLASSES == EXPECTED_CLASSES
)

model_ready = (
    model is not None
)

training_ready = (
    dataset_ready
    and model_ready
)

print("FINAL TRAINING SAFETY CHECK")

print(
    "Dataset ready:",
    dataset_ready
)

print(
    "82 classes:",
    NUM_CLASSES == EXPECTED_CLASSES
)

print(
    "Model ready:",
    model_ready
)

print(
    "Training ready:",
    training_ready
)

FINAL TRAINING SAFETY CHECK
Dataset ready: True
82 classes: True
Model ready: True
Training ready: True


##  Start YOLO Training

In [45]:
results = None

if not training_ready:

    print("TRAINING NOT STARTED")
    print("Reason: final dataset/model validation failed.")

else:

    print("YOLO CPU TRAINING STARTING")

    start_time = time.time()

    try:

        results = model.train(

            data=str(
                YOLO_CLASSIFICATION_ROOT
            ),

            imgsz=IMAGE_SIZE,

            batch=BATCH_SIZE,

            epochs=EPOCHS,

            device=DEVICE,

            workers=WORKERS,

            project=str(
                TRAIN_PROJECT
            ),

            name=TRAIN_NAME,

            pretrained=True,

            cache=False,

            amp=False,

            plots=True,

            verbose=True

        )

        elapsed_minutes = (
            time.time() - start_time
        ) / 60

        print("YOLO TRAINING COMPLETED")
        print(
            "Elapsed minutes:",
            round(elapsed_minutes, 2)
        )

    except Exception as error:

        print("YOLO TRAINING FAILED")
        print(
            type(error).__name__,
            ":",
            error
        )


YOLO CPU TRAINING STARTING
New https://pypi.org/project/ultralytics/8.4.118 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.115  Python-3.11.15 torch-2.13.0+cpu CPU (Intel Core i5-6200U 2.30GHz)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Classification_Dataset, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, im

## Best/Last Model Check

In [46]:
BEST_MODEL_PATH = (
    RUN_DIR
    / "weights"
    / "best.pt"
)

LAST_MODEL_PATH = (
    RUN_DIR
    / "weights"
    / "last.pt"
)

print("TRAINING MODEL FILES")

print("Best model:", BEST_MODEL_PATH)
print("Best exists:", BEST_MODEL_PATH.exists())

print("Last model:", LAST_MODEL_PATH)
print("Last exists:", LAST_MODEL_PATH.exists())


TRAINING MODEL FILES
Best model: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Models\classification_runs\smartagrivision_yolo_classification_cpu\weights\best.pt
Best exists: True
Last model: E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Models\classification_runs\smartagrivision_yolo_classification_cpu\weights\last.pt
Last exists: True


##  Training Results Analysis

In [47]:
RESULTS_CSV = (
    RUN_DIR
    / "results.csv"
)

if RESULTS_CSV.exists():

    training_results_df = pd.read_csv(
        RESULTS_CSV
    )

    print("TRAINING METRICS AVAILABLE")
    print("Rows:", len(training_results_df))
    print()
    display(
        training_results_df.tail()
    )

else:

    training_results_df = None

    print(
        "Training metrics are not available yet."
    )


TRAINING METRICS AVAILABLE
Rows: 3



,epoch,time,train/loss,metrics/accuracy_top1,metrics/accuracy_top5,val/loss,lr/pg0,lr/pg1,lr/pg2
0,1,4236.99,1.84761,0.93054,0.98666,0.26112,0.000058,0.000058,0.000058
1,2,6598.11,0.32559,0.96496,0.99494,0.11950,0.000078,0.000078,0.000078
2,3,8890.19,0.21410,0.96941,0.99632,0.10069,0.000039,0.000039,0.000039


In [48]:
if training_results_df is not None:

    metric_columns = [
        column
        for column in [
            "train/loss",
            "val/loss",
            "metrics/accuracy_top1",
            "metrics/accuracy_top5"
        ]
        if column in training_results_df.columns
    ]

    print("AVAILABLE YOLO CLASSIFICATION METRICS")

    for column in metric_columns:
        print(column)

else:

    print(
        "Training metrics will appear after training."
    )


AVAILABLE YOLO CLASSIFICATION METRICS
train/loss
val/loss
metrics/accuracy_top1
metrics/accuracy_top5


##  Save Training Summary

In [49]:
summary = {
    "dataset_root": str(
        YOLO_CLASSIFICATION_ROOT
    ),
    "train_images": int(
        split_summary.loc[
            split_summary["Split"] == "Train",
            "Images"
        ].iloc[0]
    ),
    "validation_images": int(
        split_summary.loc[
            split_summary["Split"] == "Validation",
            "Images"
        ].iloc[0]
    ),
    "test_images": int(
        split_summary.loc[
            split_summary["Split"] == "Test",
            "Images"
        ].iloc[0]
    ),
    "classes": int(NUM_CLASSES),
    "image_size": int(IMAGE_SIZE),
    "batch_size": int(BATCH_SIZE),
    "epochs": int(EPOCHS),
    "device": DEVICE,
    "workers": int(WORKERS),
    "model": YOLO_MODEL_NAME,
    "best_model": str(BEST_MODEL_PATH),
    "last_model": str(LAST_MODEL_PATH),
    "best_model_exists": bool(
        BEST_MODEL_PATH.exists()
    ),
    "last_model_exists": bool(
        LAST_MODEL_PATH.exists()
    )
}

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUMMARY_PATH = (
    RUN_DIR
    / "training_summary.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        summary,
        file,
        indent=4
    )

print("TRAINING SUMMARY SAVED")
print(SUMMARY_PATH)


TRAINING SUMMARY SAVED
E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Models\classification_runs\smartagrivision_yolo_classification_cpu\training_summary.json


##  Final YOLO Training Report

In [50]:
print("FINAL YOLO TRAINING REPORT")
print()
print("Train images:", summary["train_images"])
print("Validation images:", summary["validation_images"])
print("Test images:", summary["test_images"])
print("Total classes:", summary["classes"])
print("Image size:", summary["image_size"])
print("Batch size:", summary["batch_size"])
print("Epochs:", summary["epochs"])
print("Device:", summary["device"])
print("Model:", summary["model"])
print()
print("Best model exists:", summary["best_model_exists"])
print("Last model exists:", summary["last_model_exists"])
print()
print("Best model path:")
print(summary["best_model"])

if BEST_MODEL_PATH.exists():
    print()
    print("TRAINING STATUS: COMPLETED")
else:
    print()
    print("TRAINING STATUS: NOT COMPLETED")
    print(
        "The notebook is ready; run Section 16 after the "
        "dataset validation passes."
    )


FINAL YOLO TRAINING REPORT

Train images: 235731
Validation images: 29443
Test images: 29521
Total classes: 82
Image size: 128
Batch size: 32
Epochs: 3
Device: cpu
Model: yolo11n-cls.pt

Best model exists: True
Last model exists: True

Best model path:
E:\programming languages\DJANGO PROJECT\SmartAgriVision\SmartAgriVision\AgriVision\AI\notebooks\YOLO_Models\classification_runs\smartagrivision_yolo_classification_cpu\weights\best.pt

TRAINING STATUS: COMPLETED
